# Electrospray Classification Model Training
This notebook allows you to inspect, modify, and train the classification model with different feature sets.

## Cell 1: Setup & Imports

In [41]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import butter
from tqdm import tqdm

# Add project root to Python path
project_root = Path(os.getcwd()).parent
sys.path.insert(0, str(project_root))

from mapping.software.database import ElectrosprayDatabase
from mapping.software.electrospray import ElectrosprayDataProcessing
from ehda_normalization import prepare_training_data
from ehda_classifier import train

print("✓ All imports successful")

✓ All imports successful


## Cell 2: Configuration

In [42]:
# Signal Processing Parameters
SAMPLING_FREQ = 1e5
RECORD_LENGTH = 50_000
MULTIPLIER_NA = 1.0  # Data save in DB already in nA, no need to multiply
CUTOFF_HZ = 3_000

# Data Paths


# Features to exclude from training
EXCLUDE_FEATURES = [
    "actual_current_ps",  # If you find it's too noisy
    "current_PS",
    "voltage_error",
    "target_voltage",
]

# Invalid labels to filter out
INVALID_LABELS = ["undefined", "unconclusive", "noise", "", None]

print(f"Sampling frequency: {SAMPLING_FREQ} Hz")
print(f"Cutoff frequency: {CUTOFF_HZ} Hz")
print(f"Features to exclude: {EXCLUDE_FEATURES}")

Sampling frequency: 100000.0 Hz
Cutoff frequency: 3000 Hz
Features to exclude: ['actual_current_ps', 'current_PS', 'voltage_error', 'target_voltage']


## Cell 3: Helper Functions

In [43]:
def build_feature_matrix(df_db, raw_dir, sample_rate):
    """
    Iterates through DB records, loads raw .npy files, and uses the 
    ElectrosprayDataProcessing class to generate the 66-feature vector.
    """
    processing = ElectrosprayDataProcessing(sample_rate)
    
    # Filter setup (same as live)
    cutoff = CUTOFF_HZ / (0.5 * sample_rate)
    b, a = butter(6, Wn=cutoff, btype="low", analog=False)
    
    all_rows = []
    
    print(f"Extracting features from {len(df_db)} samples...")
    for _, row in tqdm(df_db.iterrows(), total=len(df_db)):
        file_path = Path(raw_dir) / str(row['raw_data_file'])
        
        if not file_path.exists():
            continue
            
        try:
            # 1. Load and Clear
            datapoints = np.load(file_path) * MULTIPLIER_NA
            processing.clear_results()
            
            # 2. Process (Matches live acquire_and_process logic)
            processing.calculate_filter(a, b, datapoints)
            processing.calculate_statistics(processing.datapoints_filtered)
            processing.calculate_power_spectral_density(processing.datapoints_filtered)
            processing.extract_advanced_ml_features()
            
            # 3. Harvest Features
            feats = processing.get_db_features_dictionary() # mean_na, etc.
            feats.update(processing.ml_features)            # advanced stats
            
            # 4. Add Metadata (Matches live classify_sample logic)
            feats.update({
                "target_voltage": float(row["target_voltage"]),
                "actual_voltage": float(row["actual_voltage"]),
                "flow_rate":      float(row["flow_rate"]),
                "voltage_error":  float(row["actual_voltage"]) - float(row["target_voltage"]),
                "current_PS":     float(row.get("actual_current_ps", 0.0)),
                "label":          row["manual_classification"]
            })
            
            all_rows.append(feats)
        except Exception as e:
            print(f"Error processing {file_path.name}: {e}")
    
    df = pd.DataFrame(all_rows)
    # 3. DROP SPECIFIC FEATURES
    # We do this before normalization so the normalizer doesn't look for them
    df = df.drop(columns=[c for c in EXCLUDE_FEATURES if c in df.columns])        
    return df

## Cell 4: Load Data from Database

In [44]:
BASE = Path(r"C:\Users\HV\Desktop\bruno_work\main\data")
db = ElectrosprayDatabase(str(BASE))
    

# 1. Load Data
df_db = db.load_training_dataframe()
print(f"✓ Loaded {len(df_db)} records from database")



[DB] Ready: C:\Users\HV\Desktop\bruno_work\main\data\data.db

--- Available Solutions in Database ---
[0] EW82
[1] EWG343
[2] EWG262
[DB] Loading samples for: ['EW82']
[DB] Loaded 1491 samples.
✓ Loaded 1491 records from database


In [45]:
df_db

,id,timestamp,solution_name,hv_position,target_voltage,actual_voltage,actual_current_ps,flow_rate,mean_na,deviation_na,...,band_power_low,band_power_mid,band_power_high,band_power_v_high,rf_spray_mode,xgb_spray_mode,image_classification,manual_classification,video_file,raw_data_file
0,1,2026-04-23T12:13:13.486060,EW82,nozzle,3000.0,3000.48,4.815040e-08,1.0,-9.475849,10.533339,...,49.701972,41.815113,5.052549,1.419719e-10,dripping (34%),dripping (81%),intermitent (100%),intermitent,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-13_486060.npy
1,2,2026-04-23T12:13:18.174207,EW82,nozzle,3200.0,3200.38,5.413520e-08,1.0,-1.104915,23.670952,...,377.396431,113.566176,38.140785,9.807355e-10,multi_jet (29%),dripping (61%),dripping (96%),dripping,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-18_174207.npy
2,3,2026-04-23T12:13:22.844423,EW82,nozzle,3400.0,3400.63,5.477000e-08,1.0,-15.125146,26.739436,...,606.451320,67.540646,16.430942,1.063364e-09,multi_jet (47%),intermitent (80%),dripping (98%),dripping,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-22_844423.npy
3,4,2026-04-23T12:13:27.489614,EW82,nozzle,3600.0,3600.50,5.368180e-08,1.0,-10.035238,31.728207,...,962.085768,18.989270,5.531595,1.380294e-09,multi_jet (49%),intermitent (86%),intermitent (80%),intermitent,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-27_489614.npy
4,5,2026-04-23T12:13:32.135037,EW82,nozzle,3800.0,3800.61,5.603950e-08,1.0,-11.275050,30.918859,...,811.633663,30.249796,5.303376,1.064484e-09,multi_jet (46%),intermitent (64%),dripping (100%),dripping,2026-04-23_12-12-47_EW82.mp4,wf_2026-04-23_12-13-32_135037.npy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1486,1794,2026-04-28T14:43:18.734449,EW82,nozzle,8300.0,8300.53,1.410960e-06,3.0,942.018748,7.476964,...,15.792868,13.649788,4.107923,6.723679e-11,multi_jet (92%),multi_jet (100%),multi_jet (100%),multi_jet,2026-04-28_14-35-41_EW82.mp4,wf_2026-04-28_14-43-18_734449.npy
1487,1795,2026-04-28T14:43:23.384156,EW82,nozzle,8500.0,8500.36,1.697050e-06,3.0,885.627466,10.752389,...,25.261421,13.094051,4.367295,6.540745e-11,multi_jet (93%),multi_jet (100%),multi_jet (100%),multi_jet,2026-04-28_14-35-41_EW82.mp4,wf_2026-04-28_14-43-23_384156.npy
1488,1796,2026-04-28T14:43:28.046817,EW82,nozzle,8700.0,8700.48,1.855650e-06,3.0,1074.457954,7.784558,...,21.608536,13.243951,3.859784,6.788967e-11,multi_jet (93%),multi_jet (100%),multi_jet (100%),multi_jet,2026-04-28_14-35-41_EW82.mp4,wf_2026-04-28_14-43-28_046817.npy
1489,1797,2026-04-28T14:43:32.703092,EW82,nozzle,8900.0,8900.08,2.157970e-06,3.0,1268.780508,7.038970,...,24.154052,13.572852,4.424740,7.753035e-11,multi_jet (93%),multi_jet (100%),multi_jet (100%),multi_jet,2026-04-28_14-35-41_EW82.mp4,wf_2026-04-28_14-43-32_703092.npy


In [46]:
# Filter samples: Keep only rows with valid manual labels
df_labeled = df_db[
    df_db['manual_classification'].notna() & 
    (~df_db['manual_classification'].isin(INVALID_LABELS))
].copy()

print(f"✓ Filtered to {len(df_labeled)} samples with valid manual labels")
print(f"  Label distribution:\n{df_labeled['manual_classification'].value_counts()}")

✓ Filtered to 1404 samples with valid manual labels
  Label distribution:
manual_classification
multi_jet      880
intermitent    377
cone_jet        77
dripping        70
Name: count, dtype: int64


## Cell 5: Build Feature Matrix

In [47]:
# Build the feature matrix from raw waveform files
df_features = build_feature_matrix(df_labeled, BASE / "raw_waveforms", SAMPLING_FREQ)

print(f"\n✓ Feature matrix built: {df_features.shape}")
print(f"  Rows (samples): {df_features.shape[0]}")
print(f"  Columns (features): {df_features.shape[1]}")

Extracting features from 1404 samples...


100%|██████████| 1404/1404 [00:11<00:00, 119.77it/s]


✓ Feature matrix built: (1404, 37)
  Rows (samples): 1404
  Columns (features): 37


## Cell 6: Inspect All Features

In [48]:
# Display all features before filtering
print("\n" + "="*80)
print("ALL AVAILABLE FEATURES BEFORE FILTERING")
print("="*80)

all_features = [col for col in df_features.columns if col != 'label']
print(f"\nTotal features: {len(all_features)}\n")

for i, feature in enumerate(all_features, 1):
    print(f"{i:3d}. {feature}")

print("\n" + "="*80)
print(f"Features scheduled for removal (EXCLUDE_FEATURES): {EXCLUDE_FEATURES}")
print("="*80)


ALL AVAILABLE FEATURES BEFORE FILTERING

Total features: 36

  1. mean_na
  2. variance_na
  3. deviation_na
  4. median_na
  5. rms_na
  6. band_power_v_low
  7. band_power_low
  8. band_power_mid
  9. band_power_high
 10. band_power_v_high
 11. peak
 12. crest_factor
 13. kurtosis
 14. skewness
 15. peak_to_peak
 16. zero_crossing_rate
 17. dominant_freq
 18. mean_freq
 19. spectral_entropy
 20. total_power
 21. wt_approx_L6_energy
 22. wt_approx_L6_energy_rel
 23. wt_detail_L6_energy
 24. wt_detail_L6_energy_rel
 25. wt_detail_L5_energy
 26. wt_detail_L5_energy_rel
 27. wt_detail_L4_energy
 28. wt_detail_L4_energy_rel
 29. wt_detail_L3_energy
 30. wt_detail_L3_energy_rel
 31. wt_detail_L2_energy
 32. wt_detail_L2_energy_rel
 33. wt_detail_L1_energy
 34. wt_detail_L1_energy_rel
 35. actual_voltage
 36. flow_rate

Features scheduled for removal (EXCLUDE_FEATURES): ['actual_current_ps', 'current_PS', 'voltage_error', 'target_voltage']


## Cell 7: Modify Features to Exclude

**Edit the `EXCLUDE_FEATURES_MODIFIED` list below to customize which features to remove:**

In [49]:
# ===== MODIFY THIS LIST =====
# Add or remove feature names as needed
EXCLUDE_FEATURES_MODIFIED = [
    "actual_current_ps",
    "current_PS",
    "voltage_error",
    "target_voltage",
    "variance_na", "rms_na", "band_power_v_low", "band_power_low", "band_power_mid", "band_power_high", "band_power_v_high", "peak", "crest_factor",
    "kurtosis", "skewness", "peak_to_peak", "zero_crossing_rate", "dominant_freq", "mean_freq", "spectral_entropy", "total_power", "wt_approx_L6_energy",
    "wt_approx_L6_energy_rel", "wt_detail_L6_energy", "wt_detail_L6_energy_rel", "wt_detail_L5_energy", "wt_detail_L5_energy_rel", "wt_detail_L4_energy",
    "wt_detail_L4_energy_rel", "wt_detail_L3_energy", "wt_detail_L3_energy_rel", "wt_detail_L2_energy", "wt_detail_L2_energy_rel", "wt_detail_L1_energy",
    "wt_detail_L1_energy_rel", "actual_voltage", "flow_rate"
    # Add more features here if you want to exclude them
    # Example: "some_feature_name",
]
# ===========================

print("\nFeatures to be EXCLUDED:")
print("-" * 40)
for feature in EXCLUDE_FEATURES_MODIFIED:
    if feature in df_features.columns:
        print(f"  ✓ {feature}")
    else:
        print(f"  ✗ {feature} (NOT FOUND)")

# Drop the excluded features
df_features = df_features.drop(
    columns=[c for c in EXCLUDE_FEATURES_MODIFIED if c in df_features.columns]
)

print(f"\nFeature matrix shape after filtering:")
print(f"  After:  {df_features.shape}")

all_features = [col for col in df_features.columns if col != 'label']
print(f"\nTotal features: {len(all_features)}\n")

for i, feature in enumerate(all_features, 1):
    print(f"{i:3d}. {feature}")


Features to be EXCLUDED:
----------------------------------------
  ✗ actual_current_ps (NOT FOUND)
  ✗ current_PS (NOT FOUND)
  ✗ voltage_error (NOT FOUND)
  ✗ target_voltage (NOT FOUND)
  ✓ variance_na
  ✓ rms_na
  ✓ band_power_v_low
  ✓ band_power_low
  ✓ band_power_mid
  ✓ band_power_high
  ✓ band_power_v_high
  ✓ peak
  ✓ crest_factor
  ✓ kurtosis
  ✓ skewness
  ✓ peak_to_peak
  ✓ zero_crossing_rate
  ✓ dominant_freq
  ✓ mean_freq
  ✓ spectral_entropy
  ✓ total_power
  ✓ wt_approx_L6_energy
  ✓ wt_approx_L6_energy_rel
  ✓ wt_detail_L6_energy
  ✓ wt_detail_L6_energy_rel
  ✓ wt_detail_L5_energy
  ✓ wt_detail_L5_energy_rel
  ✓ wt_detail_L4_energy
  ✓ wt_detail_L4_energy_rel
  ✓ wt_detail_L3_energy
  ✓ wt_detail_L3_energy_rel
  ✓ wt_detail_L2_energy
  ✓ wt_detail_L2_energy_rel
  ✓ wt_detail_L1_energy
  ✓ wt_detail_L1_energy_rel
  ✓ actual_voltage
  ✓ flow_rate

Feature matrix shape after filtering:
  After:  (1404, 4)

Total features: 3

  1. mean_na
  2. deviation_na
  3. median_na


In [6]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import butter
from tqdm import tqdm

# Add project root to Python path
project_root = Path(os.getcwd()).parent.parent
sys.path.insert(0, str(project_root))


from mapping.software.database import ElectrosprayDatabase # Import your DB class
from mapping.software.electrospray import ElectrosprayDataProcessing
from ehda_normalization import prepare_training_data
from ehda_classifier import train


SAMPLING_FREQ = 1e5
RECORD_LENGTH = 50_000
MULTIPLIER_NA = 1.0 # Data save in DB already in nA, no need to multiply
CUTOFF_HZ     = 3_000
BASE = Path(r"../data")
print(os.getcwd())
db = ElectrosprayDatabase(str(BASE))
        
    
# 1. Load Data
df_db = db.load_training_dataframe()

c:\Users\HV\Desktop\bruno_work\main\current_classification
[DB] Ready: ..\data\data.db

--- Available Solutions in Database ---
[0] NOCAP_INTER
[1] NOCAP_DRIP
[2] NOCAP_CONE
[3] NOCAP_MULTI
[4] TEST1
[DB] Loading samples for: ['NOCAP_INTER', 'TEST1']
[DB] Loaded 6380 samples.


In [7]:
df_db

,id,timestamp,solution_name,hv_position,target_voltage,actual_voltage,actual_current_ps,flow_rate,mean_na,deviation_na,...,rf_spray_mode,xgb_spray_mode,image_classification,manual_classification,video_file,raw_data_file,classical_classification,ml_classification,nn_classification,generalist_ml_classification
0,9,2026-07-02T11:37:53.302606,NOCAP_INTER,nozzle,4500.0,4500.69,1.754630e-07,15.0,34.878565,41.759450,...,intermitent (73%),intermitent (48%),N/A,intermitent,2026-07-02_11-37-20_NOCAP_INTER.mp4,wf_2026-07-02_11-37-53_302606.npy,NaN,NaN,NaN,NaN
1,10,2026-07-02T11:37:54.736171,NOCAP_INTER,nozzle,4500.0,4500.80,2.462830e-07,15.0,1.612350,2.259541,...,intermitent (59%),intermitent (36%),N/A,intermitent,2026-07-02_11-37-20_NOCAP_INTER.mp4,wf_2026-07-02_11-37-54_736171.npy,NaN,NaN,NaN,NaN
2,11,2026-07-02T11:37:56.186287,NOCAP_INTER,nozzle,4500.0,4500.78,1.744660e-07,15.0,20.714207,37.281804,...,intermitent (81%),intermitent (48%),N/A,intermitent,2026-07-02_11-37-20_NOCAP_INTER.mp4,wf_2026-07-02_11-37-56_186287.npy,NaN,NaN,NaN,NaN
3,12,2026-07-02T11:37:57.598253,NOCAP_INTER,nozzle,4500.0,4500.81,1.861630e-07,15.0,17.733067,35.390105,...,intermitent (82%),intermitent (48%),N/A,intermitent,2026-07-02_11-37-20_NOCAP_INTER.mp4,wf_2026-07-02_11-37-57_598253.npy,NaN,NaN,NaN,NaN
4,13,2026-07-02T11:37:59.009975,NOCAP_INTER,nozzle,4500.0,4500.78,1.965000e-07,15.0,2.676300,9.280399,...,intermitent (87%),intermitent (57%),N/A,intermitent,2026-07-02_11-37-20_NOCAP_INTER.mp4,wf_2026-07-02_11-37-59_009975.npy,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6375,9829,2026-07-03T11:58:34.633840,TEST1,nozzle,9000.0,9000.34,8.855690e-07,450.0,838.484715,74.290250,...,multi_jet (59%),multi_jet (40%),multi_jet (100%),N/A,2026-07-03_11-42-48_TEST1.mp4,wf_2026-07-03_11-58-34_633840.npy,cone_jet,multi_jet,multi_jet,multi_jet
6376,9830,2026-07-03T11:58:35.633301,TEST1,nozzle,9000.0,9000.34,8.110310e-07,450.0,827.813831,85.091206,...,multi_jet (59%),multi_jet (41%),multi_jet (100%),N/A,2026-07-03_11-42-48_TEST1.mp4,wf_2026-07-03_11-58-35_633301.npy,cone_jet,multi_jet,multi_jet,multi_jet
6377,9831,2026-07-03T11:58:36.639907,TEST1,nozzle,9000.0,9000.32,9.137700e-07,450.0,826.845107,87.913281,...,multi_jet (55%),multi_jet (43%),multi_jet (100%),N/A,2026-07-03_11-42-48_TEST1.mp4,wf_2026-07-03_11-58-36_639907.npy,cone_jet,multi_jet,multi_jet,multi_jet
6378,9832,2026-07-03T11:58:37.629256,TEST1,nozzle,9000.0,9000.33,8.682490e-07,450.0,832.328008,78.033491,...,multi_jet (54%),multi_jet (41%),multi_jet (100%),N/A,2026-07-03_11-42-48_TEST1.mp4,wf_2026-07-03_11-58-37_629256.npy,cone_jet,multi_jet,multi_jet,multi_jet


In [50]:
df_features['deviation_na/mean_na'] = df_features['deviation_na'] / df_features['mean_na']
df_features['mean_na/median_na'] = df_features['mean_na'] / df_features['median_na']

df_features

,mean_na,deviation_na,median_na,label,deviation_na/mean_na,mean_na/median_na
0,-9.475849,10.533339,-9.622329,intermitent,-1.111598,0.984777
1,-1.104915,23.670952,-0.931670,dripping,-21.423325,1.185951
2,-15.125146,26.739436,-15.318983,dripping,-1.767880,0.987347
3,-10.035238,31.728207,-9.705714,intermitent,-3.161679,1.033952
4,-11.275050,30.918859,-11.268562,dripping,-2.742237,1.000576
...,...,...,...,...,...,...
1399,942.018748,7.476964,942.441828,multi_jet,0.007937,0.999551
1400,885.627466,10.752389,884.734760,multi_jet,0.012141,1.001009
1401,1074.457954,7.784558,1074.227783,multi_jet,0.007245,1.000214
1402,1268.780508,7.038970,1268.782777,multi_jet,0.005548,0.999998


## Cell 8: Inspect Features After Filtering

In [51]:
# Display remaining features after filtering
print("\n" + "="*80)
print("REMAINING FEATURES AFTER FILTERING")
print("="*80)

remaining_features = [col for col in df_features.columns if col != 'label']
print(f"\nTotal features for training: {len(remaining_features)}\n")

for i, feature in enumerate(remaining_features, 1):
    print(f"{i:3d}. {feature}")

print("\n" + "="*80)
print(f"Feature statistics:")
print(f"  Min values: {df_features[remaining_features].min().min():.4f}")
print(f"  Max values: {df_features[remaining_features].max().max():.4f}")
print(f"  Missing values: {df_features[remaining_features].isna().sum().sum()}")
print("="*80)


REMAINING FEATURES AFTER FILTERING

Total features for training: 5

  1. mean_na
  2. deviation_na
  3. median_na
  4. deviation_na/mean_na
  5. mean_na/median_na

Feature statistics:
  Min values: -108.7226
  Max values: 1760.0087
  Missing values: 0


## Cell 9: Prepare Training Data

In [52]:
# Use the filtered feature matrix for training
df_norm, X, labels, feature_names, normalizer = prepare_training_data(df_features)

print(f"✓ Data prepared for training")
print(f"\nTraining data shape: {X.shape}")
print(f"  Samples: {X.shape[0]}")
print(f"  Features: {X.shape[1]}")
print(f"\nLabel distribution:")
unique, counts = np.unique(labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {label}: {count} samples")


SUCCESS Raw training data ready: 1404 samples × 5 features
  NOTE: normalizer is unfitted — it will be fitted on X_train inside train()
  Label distribution:
    cone_jet             77 samples
    dripping             70 samples
    intermitent          377 samples
    multi_jet            880 samples
✓ Data prepared for training

Training data shape: (1404, 5)
  Samples: 1404
  Features: 5

Label distribution:
  cone_jet: 77 samples
  dripping: 70 samples
  intermitent: 377 samples
  multi_jet: 880 samples


## Cell 10: Train Model

In [53]:
# Train the model with the prepared data
print("\nTraining model with modified feature set...\n")
os.chdir(r"c:\Users\HV\Desktop\bruno_work\main")
print(os.getcwd())
training_results = train(
    X=X,
    labels=labels,
    feature_names=feature_names,
    normalizer=normalizer,
    save_folder="current_classification/models"
)

print("\n✓ Model training complete!")
print(f"\nTraining results:")
for key, value in training_results.items():
    print(f"  {key}: {value}")


Training model with modified feature set...

c:\Users\HV\Desktop\bruno_work\main
  Saved label encoder and metadata to current_classification\models/

  EHDA Classifier Training
  Samples:      1404
  Features:     5
  Classes:      ['cone_jet', 'dripping', 'intermitent', 'multi_jet']
  Train/Test:   1123 / 280
    cone_jet                    77 samples  (5.5%)
    dripping                    70 samples  (5.0%)
    intermitent                377 samples  (26.9%)
    multi_jet                  880 samples  (62.7%)
SUCCESS Normalizer fitted on 1123 samples
  Strategy 1 - LINEAR (÷ factor):        0
  Strategy 2 - LOG + ROBUST:             0
  Strategy 3 - ROBUST (RobustScaler):    5
  Strategy 4 - PASSTHROUGH (unchanged):  0
  Strategy 5 - LOW-VARIANCE (unchanged): 0
SUCCESS Scalers saved to current_classification\scalers/
  Normalizer fitted on 1123 training samples -> saved to current_classification/scalers/

  Tuning hyperparameters for Random Forest...
  Best params for Random Fores

ModuleNotFoundError: No module named 'elm_study'

## Cell 11: Summary

In [ ]:
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)
print(f"Original features: {len(all_features)}")
print(f"Features used: {len(feature_names)}")
print(f"Features excluded: {len(EXCLUDE_FEATURES_MODIFIED)}")
print(f"Training samples: {X.shape[0]}")
print(f"Model save path: current_classification/models")
print("="*80)